# Planner-Driven Multitask GP Workflow (Interactive Notebook)

This notebook reproduces the end-to-end planner-driven GP workflow from `run_planned_gp_from_csv.py` in interactive cells.

- Read CSV or accept a pre-loaded DataFrame
- Configure input/output columns and planner instructions
- Run the workflow with configurable options (fallback or live planner)
- Inspect fit status, validation attempts, and compiled kernel configs

## 1. Install and Import Dependencies

In [1]:
from __future__ import annotations

import json

import pandas as pd

from bayesfolio.engine.forecast import (
    PlannedGPWorkflowOptions,
    run_planned_multitask_gp_from_dataframe,
)

## 2. Load CSV and Validate Required Columns

Set `csv_path` below to load your modeling dataframe. The cell will validate that all required columns exist before proceeding.

In [2]:
# Option 1: Load from CSV (if you have a CSV file)
# csv_path = "path/to/your/data.csv"
# df = pd.read_csv(csv_path)
# print(f"Loaded CSV: shape {df.shape}, columns: {df.columns.tolist()}")

# Option 2: Build synthetic example for testing
import numpy as np

np.random.seed(42)
rows = []
for asset in ["SPY", "VTV", "IJR"]:
    for t in range(20):
        rows.append(
            {
                "macro_1": np.random.randn() * 0.1 + t * 0.01,
                "macro_2": np.random.randn() * 0.15 + t * 0.005,
                "etf_1": np.random.randn() * 0.12 + t * 0.008,
                "y_excess_lead": np.random.randn() * 0.02 + 0.005 * (ord(asset[0]) % 3),
                "asset_id": asset,
            }
        )

df = pd.DataFrame(rows)
print(f"Created synthetic dataframe: shape {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(df.head())

Created synthetic dataframe: shape (60, 5)
Columns: ['macro_1', 'macro_2', 'etf_1', 'y_excess_lead', 'asset_id']
    macro_1   macro_2     etf_1  y_excess_lead asset_id
0  0.049671 -0.020740  0.077723       0.040461      SPY
1 -0.013415 -0.030121  0.197506       0.025349      SPY
2 -0.026947  0.091384 -0.039610       0.000685      SPY
3  0.054196 -0.271992 -0.182990      -0.001246      SPY
4 -0.061283  0.067137 -0.076963      -0.018246      SPY


## 3. Configure Input/Output Mappings

Define which columns are features (inputs), which are target/task (outputs), and free-form GP instructions.

In [3]:
# Input column configuration
input_columns = ["macro_1", "macro_2", "etf_1"]
target_column = "y_excess_lead"
task_column = "asset_id"

# Validate columns exist
missing = [c for c in input_columns + [target_column, task_column] if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

print(f"input_columns: {input_columns}")
print(f"target_column: {target_column}")
print(f"task_column: {task_column}")

# Build output columns in the required order [target, task]
output_columns = [target_column, task_column]

# Example instruction text with explicit kernel specifications
instruction_text = "Use matern 1.5 on macro and no interactions"
print(f"\ninstruction_text: {instruction_text}")

input_columns: ['macro_1', 'macro_2', 'etf_1']
target_column: y_excess_lead
task_column: asset_id

instruction_text: Use matern 1.5 on macro and no interactions


## 4. Build Planner Workflow Options

Configure runtime options: planner timeout, live endpoint enforcement, and other GP training parameters.

In [4]:
# Workflow runtime options
require_live_planner = False  # Set to True only if your OpenAI endpoint is configured
planner_timeout_seconds = 20.0  # HTTP timeout for planner calls
max_repair_steps = 8  # Max deterministic repair attempts

options = PlannedGPWorkflowOptions(
    require_live_planner=require_live_planner,
    planner_timeout_seconds=planner_timeout_seconds,
    max_repair_steps=max_repair_steps,
)

print("Planner options:")
print(f"  require_live_planner={require_live_planner}")
print(f"  planner_timeout_seconds={planner_timeout_seconds}")
print(f"  max_repair_steps={max_repair_steps}")

Planner options:
  require_live_planner=False
  planner_timeout_seconds=20.0
  max_repair_steps=8


## 5. Run `run_planned_multitask_gp_from_dataframe`

Execute the end-to-end workflow. This may take a few seconds while building, fitting, and validating the GP.

In [5]:
print("Running planner-driven multitask GP workflow...\n")

artifacts = run_planned_multitask_gp_from_dataframe(
    df=df,
    input_columns=input_columns,
    output_columns=output_columns,
    instruction_text=instruction_text,
    options=options,
)

print("Workflow completed successfully!")

Running planner-driven multitask GP workflow...



ValueError: Planner block macro references unknown groups: ['macro']

## 6. Inspect Status and Validation Attempts

Check the final status, client status (indicating whether the planner was live or fell back), and how many fit/repair attempts were made.

In [ ]:
print("=== Workflow Results ===\n")

print(f"final_status: {artifacts.result.final_status}")
print(f"planner_client_status: {artifacts.result.planner_client_status}")
print(f"attempt_count: {artifacts.result.fit_validation.attempt_count}")
print(f"build_success: {artifacts.result.fit_validation.build_success}")
print(f"fit_success: {artifacts.result.fit_validation.fit_success}")
print(f"prediction_success: {artifacts.result.fit_validation.prediction_success}")

if artifacts.result.diagnostics:
    print(f"\nDiagnostics ({len(artifacts.result.diagnostics)} messages):")
    for msg in artifacts.result.diagnostics:
        print(f"  - {msg}")

if artifacts.result.repair_attempts:
    print(f"\nRepair Attempts ({len(artifacts.result.repair_attempts)}):")
    for attempt in artifacts.result.repair_attempts:
        print(f"  Step {attempt.step}: {attempt.action} -> {attempt.status}")
        if attempt.detail:
            print(f"           {attempt.detail}")

## 7. Render Mean/Covariance Config JSON

Pretty-print the compiled mean and covariance configurations passed to the active multitask GP builder.

In [ ]:
print("=== Mean Module Config ===\n")
print(json.dumps(artifacts.result.mean_config, indent=2, sort_keys=True))

In [ ]:
print("\n=== Covariance Module Config ===\n")
print(json.dumps(artifacts.result.covar_config, indent=2, sort_keys=True))

## 8. Reusable Notebook Function

Wrap the entire workflow in a function that you can call with different datasets and instructions.

In [ ]:
def run_planned_gp_notebook(
    df: pd.DataFrame,
    input_columns: list[str],
    target_column: str,
    task_column: str,
    instruction_text: str,
    require_live_planner: bool = False,
    planner_timeout_seconds: float = 20.0,
):
    """
    Run the planner-driven GP workflow and return artifacts.

    Args:
        df: Input dataframe with all feature, target, and task columns.
        input_columns: List of usable non-task feature column names.
        target_column: Name of the target return column (decimal units).
        task_column: Name of the task/asset identifier column.
        instruction_text: Free-form GP instruction text.
        require_live_planner: Whether to enforce live planner mode.
        planner_timeout_seconds: HTTP timeout for planner calls.

    Returns:
        artifacts: PlannedMultitaskGPArtifacts with result, model, and metadata.
    """
    output_columns = [target_column, task_column]
    options = PlannedGPWorkflowOptions(
        require_live_planner=require_live_planner,
        planner_timeout_seconds=planner_timeout_seconds,
    )
    return run_planned_multitask_gp_from_dataframe(
        df=df,
        input_columns=input_columns,
        output_columns=output_columns,
        instruction_text=instruction_text,
        options=options,
    )


# Example: Run again with different instruction text
if False:  # Set to True to re-run with a new instruction
    new_instruction = "Use a matern kernel with ard for all input variables"
    new_artifacts = run_planned_gp_notebook(
        df=df,
        input_columns=input_columns,
        target_column=target_column,
        task_column=task_column,
        instruction_text=new_instruction,
        require_live_planner=False,
    )
    print(f"New run final_status: {new_artifacts.result.final_status}")